# Praktikum _Vowel ASR_: DTW vs Gaussian

Notebook ini menjadi antarmuka eksplorasi dan visualisasi perbandingan ASR huruf vokal antara metode DTW dengan Gaussian.

Jalankan dari root repositori atau isi `PROJECT_ROOT` di bawah.


## Struktur data yang didukung

```text
data/template/<speaker>/*.wav       # train dan closed recordings
data/test/closed/<speaker>/*.wav    # closed speakers
data/test/open/<vowel>/*.wav       # open; folder A/E/I/O/U adalah label vowel
```

Nama file yang didukung antara lain `S01_a.wav`, `G09_S01_A01.wav`, dan `S2_Closed_i.wav`. Label open terutama diambil dari folder vowel; identitas speaker open diambil dari nama file.


## Metode

- Fitur: MFCC 39 dimensi (13 MFCC + delta + delta-delta), frame 25 ms dan langkah 10 ms.
- DTW: jarak Euclidean per frame, tiga langkah standar, biaya dibagi panjang alignment; template memakai rekaman berpanjang median.
- Gaussian: satu Gaussian full-covariance per vowel dari seluruh frame latih; prediksi memakai negative log-likelihood rata-rata.
- Audio stereo dikonversi ke mono dan sample rate selain 16 kHz di-resample sebelum MFCC.


In [11]:
from pathlib import Path
import sys
from IPython.display import display, Markdown

# Isi dengan Path(r"C:/path/ke/repositori") bila notebook dijalankan dari luar repo.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    ROOT = next((p for p in candidates if (p / "evaluation.py").is_file() and (p / "data" / "template").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Root proyek tidak ditemukan; isi PROJECT_ROOT.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation import (
    find_project_root, run_experiment, dataset_stats, summary_rows,
    per_vowel_rows, per_speaker_rows, confusion_matrix,
)
from preprocessing import VOWELS
from reporting import format_markdown_table, save_predictions

ROOT = find_project_root(project_root=ROOT)
experiment = run_experiment(ROOT)
results = experiment["results"]
scenarios = list(experiment["scenarios"])
systems = experiment["systems"]
print(f"Root proyek: {ROOT}")


Root proyek: C:\Users\Jovan\Documents\Projects\vowel-asr-dtw-vs-gaussian


In [12]:
# Ringkasan distribusi data dan akurasi tiap skenario.
stats_rows = [[name, *dataset_stats(records).values()] for name, records in experiment["raw"].items()]
display(Markdown(format_markdown_table(
    ["Skenario", "WAV", "Grup/speaker", "Distribusi vowel", "Sample rate", "Kanal", "Tipe"], stats_rows, percent=False
)))

summary_scenarios, rows = summary_rows(results, systems, scenarios)
display(Markdown(format_markdown_table(
    ["Sistem", *summary_scenarios, "Rata-rata skenario"], rows
)))


| Skenario | WAV | Grup/speaker | Distribusi vowel | Sample rate | Kanal | Tipe |
| --- | --- | --- | --- | --- | --- | --- |
| Closed Recording | 25 | 5 | {'a': 5, 'e': 5, 'i': 5, 'o': 5, 'u': 5} | [16000, 48000] | [1, 2] | ['int16'] |
| Closed Speakers | 25 | 5 | {'a': 5, 'e': 5, 'i': 5, 'o': 5, 'u': 5} | [16000, 48000] | [1, 2] | ['int16'] |
| Open Speakers | 85 | 17 | {'a': 17, 'e': 17, 'i': 17, 'o': 17, 'u': 17} | [16000, 48000] | [1] | ['int16'] |

| Sistem | Closed Recording | Closed Speakers | Open Speakers | Rata-rata skenario |
| --- | --- | --- | --- | --- |
| DTW | 100,00% | 88,00% | 67,06% | 85,02% |
| Gaussian | 100,00% | 76,00% | 40,00% | 72,00% |

In [13]:
# Akurasi per vowel dan per speaker/group.
columns, rows = per_vowel_rows(results, systems, scenarios)
display(Markdown(format_markdown_table(
    ["Vowel aktual", *[f"{system} {scenario}" for system, scenario in columns]], rows
)))

for selected, title in [
    (["Closed Recording", "Closed Speakers"], "Akurasi per speaker : template dan closed"),
    (["Open Speakers"], "Akurasi per speaker ? open"),
]:
    selected = [scenario for scenario in selected if scenario in scenarios]
    if selected:
        columns, rows = per_speaker_rows(results, selected, systems)
        display(Markdown(f"### {title}\n" + format_markdown_table(
            ["Speaker/folder", *[f"{system} {scenario}" for system, scenario in columns], "Rata-rata"], rows
        )))


| Vowel aktual | DTW Closed Recording | DTW Closed Speakers | DTW Open Speakers | Gaussian Closed Recording | Gaussian Closed Speakers | Gaussian Open Speakers |
| --- | --- | --- | --- | --- | --- | --- |
| a | 100,00% | 100,00% | 52,94% | 100,00% | 100,00% | 88,24% |
| i | 100,00% | 100,00% | 82,35% | 100,00% | 100,00% | 70,59% |
| u | 100,00% | 80,00% | 47,06% | 100,00% | 40,00% | 11,76% |
| e | 100,00% | 60,00% | 82,35% | 100,00% | 80,00% | 17,65% |
| o | 100,00% | 100,00% | 70,59% | 100,00% | 60,00% | 11,76% |
| Semua vowel | 100,00% | 88,00% | 67,06% | 100,00% | 76,00% | 40,00% |

### Akurasi per speaker : template dan closed
| Speaker/folder | DTW Closed Recording | DTW Closed Speakers | Gaussian Closed Recording | Gaussian Closed Speakers | Rata-rata |
| --- | --- | --- | --- | --- | --- |
| danen | 100,00% | 60,00% | 100,00% | 60,00% | 80,00% |
| farrel | 100,00% | 100,00% | 100,00% | 100,00% | 100,00% |
| jovan | 100,00% | 100,00% | 100,00% | 80,00% | 95,00% |
| kenan | 100,00% | 100,00% | 100,00% | 80,00% | 95,00% |
| lukas | 100,00% | 80,00% | 100,00% | 60,00% | 85,00% |
| Rata-rata speaker | 100,00% | 88,00% | 100,00% | 76,00% | 91,00% |

### Akurasi per speaker ? open
| Speaker/folder | DTW Open Speakers | Gaussian Open Speakers | Rata-rata |
| --- | --- | --- | --- |
| G04_S03 | 60,00% | 40,00% | 50,00% |
| G04_S04 | 40,00% | 20,00% | 30,00% |
| G07_S01 | 80,00% | 40,00% | 60,00% |
| G07_S02 | 40,00% | 60,00% | 50,00% |
| G07_S03 | 80,00% | 40,00% | 60,00% |
| G07_S04 | 80,00% | 40,00% | 60,00% |
| G07_S05 | 60,00% | 40,00% | 50,00% |
| G09_S01 | 80,00% | 60,00% | 70,00% |
| G09_S02 | 80,00% | 60,00% | 70,00% |
| G09_S03 | 80,00% | 40,00% | 60,00% |
| G09_S04 | 60,00% | 20,00% | 40,00% |
| G09_S05 | 80,00% | 40,00% | 60,00% |
| Kel3_S01 | 80,00% | 20,00% | 50,00% |
| Kel3_S02 | 60,00% | 40,00% | 50,00% |
| Kel3_S03 | 40,00% | 20,00% | 30,00% |
| Kel3_S05 | 40,00% | 20,00% | 30,00% |
| S2_Closed | 100,00% | 80,00% | 90,00% |
| Rata-rata speaker | 67,06% | 40,00% | 53,53% |

In [14]:
# Confusion matrix: baris = vowel aktual, kolom = vowel prediksi.
CONFUSION_SCENARIO = "Open Speakers"
if not experiment["raw"].get(CONFUSION_SCENARIO):
    print(f"{CONFUSION_SCENARIO}: tidak ada data WAV; confusion matrix dilewati.")
else:
    for system in systems:
        matrix = confusion_matrix(results[(system, CONFUSION_SCENARIO)])
        display(Markdown(f"### {system} : {CONFUSION_SCENARIO}\n" + format_markdown_table(
            ["Aktual \\\ Prediksi", *[vowel.upper() for vowel in VOWELS]],
            [[vowel.upper(), *matrix[index].tolist()] for index, vowel in enumerate(VOWELS)],
            percent=False,
        )))


### DTW : Open Speakers
| Aktual \\ Prediksi | A | I | U | E | O |
| --- | --- | --- | --- | --- | --- |
| A | 9 | 2 | 1 | 3 | 2 |
| I | 0 | 14 | 0 | 3 | 0 |
| U | 0 | 0 | 8 | 0 | 9 |
| E | 0 | 2 | 0 | 14 | 1 |
| O | 2 | 0 | 3 | 0 | 12 |

### Gaussian : Open Speakers
| Aktual \\ Prediksi | A | I | U | E | O |
| --- | --- | --- | --- | --- | --- |
| A | 15 | 2 | 0 | 0 | 0 |
| I | 5 | 12 | 0 | 0 | 0 |
| U | 9 | 1 | 2 | 0 | 5 |
| E | 10 | 4 | 0 | 3 | 0 |
| O | 15 | 0 | 0 | 0 | 2 |

## Panduan Analisis hasil

Gunakan tabel untuk membandingkan closed recordings, closed speakers, dan open. _Closed recordings_ memakai data yang sama dengan template sehingga tidak mencerminkan generalisasi. Pada skenario _open_, _ground truth vowel_ berasal dari folder (`open/A`, `open/E`, dan seterusnya). Identitas speaker tidak digunakan sebagai kelas yang diprediksi. DTW mempertahankan perbandingan urutan temporal, sedangkan Gaussian merangkum distribusi _frame_ tanpa urutan.

In [15]:
# Simpan prediksi sehingga hasil dapat ditinjau tanpa menjalankan ulang evaluasi.
out_path = save_predictions(results, ROOT)
print(f"Prediksi disimpan: {out_path.relative_to(ROOT).as_posix()} ({sum(map(len, results.values()))} baris)")


Prediksi disimpan: outputs/predictions.csv (270 baris)
